# Intro 07 — Joint Distributions, Covariance, and Correlation

Practice notebook for the **"Joint Distributions, Covariance, and Correlation"** section of the stats course PDF.

## What you will learn

This notebook recaps the theory from the PDF section, then **verifies it with Python**.
Each part ends with a **"Your turn"** prompt; the full **Problem Set** at the end has
**click-to-reveal solutions**.

## How to use

1. **Read** each markdown cell, then **run** the code beneath it (`Shift+Enter`).
2. **Change parameters** and re-run — statistics is about *relationships*, not memorized formulas.
3. LaTeX uses \( \) for inline math and \[ \] / $$ $$ for display math (KaTeX-friendly).

Let's begin.


---
## Setup — run this first

NumPy for numerics, SciPy for exact distributions, Matplotlib for plots.


In [ ]:
# If anything is missing, uncomment and run once:
# !pip install numpy scipy matplotlib

import numpy as np
import matplotlib.pyplot as plt
import scipy
from scipy import stats

np.random.seed(42)
rng = np.random.default_rng(42)
plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.grid"] = True

print("Ready. NumPy", np.__version__, "| SciPy", scipy.__version__)


---
## Part 1 — Joint pmf and marginals

For two discrete random variables \(X\) and \(Y\), the **joint pmf** is

$$
p_{X,Y}(x,y) = P(X=x,\, Y=y).
$$

The **marginal** distributions sum out the other variable:

$$
p_X(x) = \sum_{y} p_{X,Y}(x,y),
\qquad
p_Y(y) = \sum_{x} p_{X,Y}(x,y).
$$

The PDF works a small example: \(X\) = number of daily trades \(\in\{0,1,2\}\),
\(Y\) = end-of-day sign \(\in\{0,1\}\), with joint pmf

| \(Y \backslash X\) | 0 | 1 | 2 |
|---|---|---|---|
| 0 | 0.10 | 0.15 | 0.10 |
| 1 | 0.20 | 0.25 | 0.20 |

For instance \(p_X(1) = p_{X,Y}(1,0) + p_{X,Y}(1,1) = 0.15 + 0.25 = 0.40\).
Let's build this table as a NumPy array and verify the marginals.


In [ ]:
# Joint pmf table: rows index Y in {0,1}, columns index X in {0,1,2}
P = np.array([
    [0.10, 0.15, 0.10],   # Y = 0
    [0.20, 0.25, 0.20],   # Y = 1
])
x_vals = np.array([0, 1, 2])
y_vals = np.array([0, 1])

# Sanity check: a joint pmf must sum to 1
assert np.isclose(P.sum(), 1.0)
print(f"sum(P) = {P.sum():.2f}  (a valid joint pmf)")

# Marginals: sum over the other axis
p_X = P.sum(axis=0)   # sum over Y (rows) -> shape (3,)
p_Y = P.sum(axis=1)   # sum over X (cols) -> shape (2,)

print("p_X =", p_X, " sum =", p_X.sum())
print("p_Y =", p_Y, " sum =", p_Y.sum())

# The PDF's hand calculation: p_X(1) = 0.15 + 0.25 = 0.40
print(f"p_X(1) = {p_X[1]:.2f}  (expected 0.40)")
assert np.isclose(p_X[1], 0.40)

# Quick visualization of the joint pmf
fig, ax = plt.subplots(figsize=(7, 4))
im = ax.imshow(P, cmap="Blues", aspect="auto")
ax.set_xticks(x_vals); ax.set_xticklabels(x_vals)
ax.set_yticks(y_vals); ax.set_yticklabels(y_vals)
ax.set_xlabel("X (number of trades)")
ax.set_ylabel("Y (end-of-day sign)")
ax.set_title("Joint pmf $p_{X,Y}(x,y)$")
for i in y_vals:
    for j in x_vals:
        ax.text(j, i, f"{P[i, j]:.2f}", ha="center", va="center",
                color="white" if P[i, j] > 0.15 else "black")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout(); plt.show()


---
## Part 2 — Expectations and covariance from the joint table

The **covariance** between \(X\) and \(Y\) is

$$
\operatorname{Cov}(X,Y) = E\!\left[(X - E[X])(Y - E[Y])\right]
       = E[XY] - E[X]\,E[Y].
$$

It is positive when \(X\) and \(Y\) tend to move together, negative when they
move in opposite directions, and zero when they are **uncorrelated** (which is
*not* the same as independent).

For a discrete joint distribution,

$$
E[g(X,Y)] = \sum_{x}\sum_{y} g(x,y)\, p_{X,Y}(x,y).
$$

So \(E[X] = \sum_x x\, p_X(x)\), \(E[Y] = \sum_y y\, p_Y(y)\), and
\(E[XY] = \sum_x\sum_y x\, y\, p_{X,Y}(x,y)\). Let's compute all of these
straight from the joint table and compare with the definition
\(\operatorname{Cov}(X,Y) = E[(X-E[X])(Y-E[Y])]\).


In [ ]:
# Expectations directly from the joint table
EX = np.sum(x_vals * p_X)
EY = np.sum(y_vals * p_Y)
EXY = np.sum(np.outer(y_vals, x_vals) * P)   # sum over all (y, x) of x*y*p(x,y)

cov_def = np.sum(np.outer(y_vals - EY, x_vals - EX) * P)   # E[(X-EX)(Y-EY)]
cov_short = EXY - EX * EY                                  # E[XY] - E[X]E[Y]

print(f"E[X]   = {EX:.4f}")
print(f"E[Y]   = {EY:.4f}")
print(f"E[XY]  = {EXY:.4f}")
print(f"Cov(X, Y) via definition   = {cov_def:.4f}")
print(f"Cov(X, Y) via E[XY]-E[X]E[Y] = {cov_short:.4f}")
assert np.isclose(cov_def, cov_short)
print("Match: both covariance formulas agree ✔")

# Sign interpretation
sign = "positive" if cov_def > 0 else ("negative" if cov_def < 0 else "zero")
print(f"Cov is {sign}  ->  X and Y tend to move in the same direction.")


---
## Part 3 — Correlation: a dimensionless measure

The **correlation coefficient** normalizes covariance:

$$
\rho_{X,Y} = \frac{\operatorname{Cov}(X,Y)}
                  {\sqrt{\operatorname{Var}(X)\,\operatorname{Var}(Y)}}.
$$

It is a dimensionless number in \([-1, 1]\) that measures **linear** association.
The PDF's asset-returns example notes that \(\rho \approx 0.9\) means two stocks
move together strongly, \(\rho \approx -0.5\) means one rises as the other
falls, and \(\rho \approx 0\) means *no linear* relationship (nonlinear
dependencies may remain).


In [ ]:
# Variances from the marginals
VarX = np.sum((x_vals - EX) ** 2 * p_X)
VarY = np.sum((y_vals - EY) ** 2 * p_Y)

rho = cov_def / np.sqrt(VarX * VarY)

print(f"Var(X) = {VarX:.4f}")
print(f"Var(Y) = {VarY:.4f}")
print(f"rho    = {rho:.4f}   (in [-1, 1]: {(-1 <= rho <= 1)})")

# Cross-check: sample the joint distribution many times and compare empirical
# cov / corr to the theoretical values above.
rng = np.random.default_rng(7)
N = 200_000
# Sample (X, Y) pairs from the joint pmf by flattening it as a categorical
flat_p = P.ravel()
idx = rng.choice(len(flat_p), size=N, p=flat_p)
Yi, Xi = np.divmod(idx, P.shape[1])   # row = Y, col = X

cov_sample = np.cov(Xi, Yi, ddof=0)[0, 1]
corr_sample = np.corrcoef(Xi, Yi)[0, 1]
print(f"sample Cov(X, Y) = {cov_sample:.4f}  (theory {cov_def:.4f})")
print(f"sample corr      = {corr_sample:.4f}  (theory {rho:.4f})")
assert np.isclose(cov_sample, cov_def, atol=2e-2)
assert np.isclose(corr_sample, rho, atol=2e-2)
print("Match: sample estimates track the theoretical cov/corr ✔")


---
## Part 4 — Simulating correlated normals via Cholesky

A standard way to build a pair \((R_1, R_2)\) with a target correlation
\(\rho\) is to start with independent standard normals \(Z_1, Z_2\) and apply
the Cholesky factor of the target covariance matrix

$$
\Sigma = \begin{pmatrix} \sigma_1^2 & \rho\,\sigma_1\sigma_2 \\[2pt]
                          \rho\,\sigma_1\sigma_2 & \sigma_2^2 \end{pmatrix}
= L\,L^{\top}.
$$

Then \(\mathbf{R} = L\,\mathbf{Z}\) has covariance \(\Sigma\). We simulate this
for \(\rho = 0.9\) and \(\rho = -0.5\) and compare the sample covariance and
correlation to the theoretical values.


In [ ]:
def simulate_correlated_normals(rho, n=20_000, sigma1=1.0, sigma2=1.0, seed=42):
    '''Return (R1, R2) with target correlation rho via Cholesky.'''
    Sigma = np.array([[sigma1**2, rho * sigma1 * sigma2],
                      [rho * sigma1 * sigma2, sigma2**2]])
    L = np.linalg.cholesky(Sigma)
    rng = np.random.default_rng(seed)
    Z = rng.standard_normal(size=(2, n))      # two independent standard normals
    R = L @ Z                                  # apply Cholesky factor
    return R[0], R[1], Sigma, L

for rho_true in [0.9, -0.5, 0.0]:
    R1, R2, Sigma, L = simulate_correlated_normals(rho_true)
    cov_s = np.cov(R1, R2, ddof=0)[0, 1]
    rho_s = np.corrcoef(R1, R2)[0, 1]
    print(f"rho_true = {rho_true:+.2f}  | "
          f"sample cov = {cov_s:+.4f} (theory {Sigma[0,1]:+.4f}) | "
          f"sample rho = {rho_s:+.4f}")
    assert np.isclose(rho_s, rho_true, atol=2e-2)

# Scatter plots for the three correlations
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, rho_true in zip(axes, [0.9, -0.5, 0.0]):
    R1, R2, _, _ = simulate_correlated_normals(rho_true, seed=42)
    ax.scatter(R1[:3000], R2[:3000], s=6, alpha=0.4)
    ax.set_xlabel("R1"); ax.set_ylabel("R2")
    ax.set_title(rf"$\rho = {rho_true:+.1f}$ (sample $\hat\rho = {np.corrcoef(R1,R2)[0,1]:+.3f}$)")
plt.tight_layout(); plt.show()
print("Cholesky simulation reproduces the target correlations ✔")


---
## Part 5 — Correlation zero does **not** imply independence

The PDF warns: \(\operatorname{Cov}(X,Y) = 0\) means **uncorrelated**, not
**independent**. A classic counterexample is a nonlinear deterministic
relationship \(Y = X^2\) with \(X\) symmetric about zero. Then

$$
\operatorname{Cov}(X, Y) = E[X\cdot X^2] - E[X]\,E[X^2] = E[X^3] - 0 \cdot E[X^2].
$$

If \(X\) has a symmetric distribution about \(0\), \(E[X^3] = 0\), so
\(\operatorname{Cov}(X, Y) = 0\) and \(\rho_{X,Y} = 0\) — yet \(Y\) is a
deterministic function of \(X\). We demonstrate this numerically with
\(X \sim \mathcal{N}(0,1)\), \(Y = X^2\).


In [ ]:
rng = np.random.default_rng(123)
n = 50_000
X = rng.standard_normal(size=n)
Y = X**2   # deterministic, nonlinear function of X

cov_xy = np.cov(X, Y, ddof=0)[0, 1]
corr_xy = np.corrcoef(X, Y)[0, 1]

print(f"Cov(X, X^2)  = {cov_xy:+.4f}   (theory 0 by symmetry)")
print(f"corr(X, X^2) = {corr_xy:+.4f}   (theory 0)")
assert np.isclose(cov_xy, 0.0, atol=3e-2)
assert np.isclose(corr_xy, 0.0, atol=3e-2)
print("Linear correlation is ~0, yet Y = X^2 is a PERFECT function of X.")

# Visualize: scatter is a parabola, but the linear fit is flat.
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].scatter(X[:4000], Y[:4000], s=6, alpha=0.4, color="steelblue")
axes[0].set_xlabel("X"); axes[0].set_ylabel(r"$Y = X^2$")
axes[0].set_title(rf"$\rho = {corr_xy:+.3f}$: no *linear* association")

# Best linear fit (slope ~ 0 even though Y is determined by X)
b1 = np.cov(X, Y, ddof=0)[0, 1] / np.var(X)
b0 = Y.mean() - b1 * X.mean()
xs = np.linspace(X.min(), X.max(), 100)
axes[0].plot(xs, b0 + b1 * xs, color="crimson", lw=2,
             label=rf"linear fit: $\hat y = {b0:.2f} + {b1:+.3f}\,x$")
axes[0].legend()

# A nonlinear (quadratic) fit recovers the relationship
b2, b1q, b0q = np.polyfit(X, Y, deg=2)
axes[1].scatter(X[:4000], Y[:4000], s=6, alpha=0.4, color="steelblue")
axes[1].plot(xs, b0q + b1q * xs + b2 * xs**2, color="crimson", lw=2,
             label=rf"quad fit: $\hat y = {b0q:.2f} + {b1q:+.2f}\,x + {b2:.2f}\,x^2$")
axes[1].set_xlabel("X"); axes[1].set_ylabel(r"$Y = X^2$")
axes[1].set_title("A quadratic fit recovers the deterministic relation")
axes[1].legend()

plt.tight_layout(); plt.show()
print(f"Quadratic fit coefficients: b0={b0q:.3f}, b1={b1q:.3f}, b2={b2:.3f}")
print("(Expected roughly b0=1, b1=0, b2=1 for Y = X^2 with E[X^2]=1, E[X^3]=0.)")


---
**Your turn:**

- Modify the joint pmf in Part 1 so that \(X\) and \(Y\) are **independent**:
  set \(p_{X,Y}(x,y) = p_X(x)\, p_Y(y)\) using the marginals you computed.
  Recompute \(\operatorname{Cov}(X,Y)\). Is it zero? It should be — independence
  implies zero covariance.
- In Part 4, push \(\rho\) toward its boundaries. Try \(\rho = 0.99\) and
  \(\rho = -0.99\). Does the sample correlation still track the target? What
  happens to the Cholesky factor as \(\rho \to \pm 1\)?
- In Part 5, replace \(Y = X^2\) with \(Y = X^3\). Now \(E[X^4] \ne 0\), so
  \(\operatorname{Cov}(X, Y) = E[X^4] \ne 0\). Compute the correlation and
  explain why the cubic case is *not* a counterexample to "corr=0 ⟹ independent".
- Build a 3-variable joint table (e.g. trades \(X\), sign \(Y\), and a third
  binary variable \(W\)) and compute all three pairwise covariances. Which
  pair is most strongly correlated?


---
# Problem Set

**Try each problem before revealing the solution.**


## Problems

1. Using the joint pmf from Part 1, compute \(E[X]\), \(E[Y]\),
\(\operatorname{Var}(X)\), \(\operatorname{Var}(Y)\), \(\operatorname{Cov}(X,Y)\),
and \(\rho_{X,Y}\) **by hand** from the marginals and the joint table, then
verify each value in Python.

2. Construct two **independent** discrete random variables (a joint pmf
that factors as \(p_X(x)\,p_Y(y)\)) and show numerically that
\(\operatorname{Cov}(X,Y) = 0\). Then confirm the converse is *false* by
re-using the \(Y = X^2\) example: compute \(\operatorname{Cov}(X, Y)\) and
check that \(p_{X,Y}(x,y) \ne p_X(x)\,p_Y(y)\) for some \((x, y)\).

3. Simulate a bivariate normal with \(\rho = 0.6\), \(\sigma_1 = 2\),
\(\sigma_2 = 3\) using the Cholesky method of Part 4. Report the sample
covariance matrix and compare each entry to the theoretical \(\Sigma\). Why
does the off-diagonal entry equal \(\rho\,\sigma_1\sigma_2\)?

4. For \(X \sim \mathcal{N}(0,1)\) and \(Y = X^2\), show analytically that
\(\operatorname{Cov}(X, Y) = E[X^3] = 0\) and confirm numerically that the
linear correlation is approximately \(0\) while the mutual information-style
check \(P(|Y - X^2| < \epsilon) = 1\) (for tiny \(\epsilon\)) holds — i.e.
\(Y\) is a deterministic function of \(X\).

5. Write a function `joint_moments(P, x_vals, y_vals)` that takes a joint
pmf array `P` (rows = \(Y\), cols = \(X\)) and returns a dict with keys
`EX`, `EY`, `VarX`, `VarY`, `Cov`, `rho`. Test it on the trades/sign table from
the PDF and on an independent joint pmf of your choice.


<details>
<summary><strong>Reveal solutions</strong></summary>

**1.** From Part 1, \(p_X = (0.30,\, 0.40,\, 0.30)\) and
\(p_Y = (0.35,\, 0.65)\). So
\(E[X] = 0(0.30) + 1(0.40) + 2(0.30) = 1.0\),
\(E[Y] = 0(0.35) + 1(0.65) = 0.65\).
\(E[X^2] = 0^2(0.30) + 1^2(0.40) + 2^2(0.30) = 1.6\),
\(\operatorname{Var}(X) = 1.6 - 1.0^2 = 0.6\).
\(\operatorname{Var}(Y) = 0.65 - 0.65^2 = 0.2275\).
\(E[XY] = \sum x\, y\, p_{X,Y}(x,y)\) — only \(y=1\) rows contribute:
\(E[XY] = 0(0.20) + 1(0.25) + 2(0.20) = 0.65\).
\(\operatorname{Cov}(X,Y) = 0.65 - (1.0)(0.65) = 0.0\).
\(\rho = 0 / \sqrt{0.6 \cdot 0.2275} = 0\). (Here trades and sign are
uncorrelated — but the table was not built to be independent, just zero-cov.)

```python
P = np.array([[0.10,0.15,0.10],[0.20,0.25,0.20]])
xv = np.array([0,1,2]); yv = np.array([0,1])
pX = P.sum(0); pY = P.sum(1)
EX = (xv*pX).sum(); EY = (yv*pY).sum()
VarX = ((xv-EX)**2*pX).sum(); VarY = ((yv-EY)**2*pY).sum()
EXY = (np.outer(yv, xv)*P).sum()
Cov = EXY - EX*EY
rho = Cov/np.sqrt(VarX*VarY)
print(EX, EY, VarX, VarY, Cov, rho)
# 1.0 0.65 0.6 0.2275 0.0 0.0
```

**2.** Independence: take \(p_X = (0.5, 0.5)\), \(p_Y = (0.4, 0.6)\) and
\(P = p_X \otimes p_Y\) (outer product). Then \(\operatorname{Cov}=0\). For
the converse, with \(X\sim\mathcal N(0,1)\), \(Y=X^2\): the sample covariance is
\(\approx 0\) but \(P(-1 < X < 1,\, 0 \leq Y < 1)\) is not equal to
\(P(-1 < X < 1)\,P(0 \leq Y < 1)\) — knowing \(X\) constrains \(Y\), so they are
dependent despite zero covariance.

```python
rng = np.random.default_rng(0)
X = rng.standard_normal(50000); Y = X**2
print("Cov(X, Y)   =", np.cov(X, Y, ddof=0)[0,1])       # ~ 0
# Check dependence: P(-1<X<1, 0<=Y<1) vs product of marginals
a = ((X>-1)&(X<1)&(Y>=0)&(Y<1)).mean()
b = ((X>-1)&(X<1)).mean() * ((Y>=0)&(Y<1)).mean()
print("joint =", a, " product of marginals =", b)      # not equal
```

**3.** With \(\rho = 0.6\), \(\sigma_1 = 2\), \(\sigma_2 = 3\),

$$
\Sigma = \begin{pmatrix}4 & 0.6\cdot 2\cdot 3 \\ 0.6\cdot 2\cdot 3 & 9\end{pmatrix}
      = \begin{pmatrix}4 & 3.6 \\ 3.6 & 9\end{pmatrix}.
$$

The off-diagonal is \(\rho\,\sigma_1\sigma_2\) because
\(\operatorname{Cov}(R_1, R_2) = \rho\,\sigma_1\sigma_2\) by definition of
\(\rho = \operatorname{Cov}/(\sigma_1\sigma_2)\).

```python
rho, s1, s2 = 0.6, 2.0, 3.0
Sigma = np.array([[s1**2, rho*s1*s2],[rho*s1*s2, s2**2]])
L = np.linalg.cholesky(Sigma)
rng = np.random.default_rng(42)
Z = rng.standard_normal((2, 20000)); R = L @ Z
print(np.cov(R[0], R[1], ddof=0))      # ~ [[4, 3.6],[3.6, 9]]
print("theory off-diag =", rho*s1*s2) # 3.6
```

**4.** For \(X\sim\mathcal N(0,1)\), every odd central moment is \(0\), so
\(E[X^3] = 0\). Then
\(\operatorname{Cov}(X, Y) = E[X\cdot X^2] - E[X]E[X^2] = E[X^3] - 0 = 0\).
But \(Y = X^2\) is deterministic: \(P(|Y - X^2| < 10^{-9}) = 1\). Zero correlation,
perfect (nonlinear) dependence.

```python
rng = np.random.default_rng(2024)
X = rng.standard_normal(100000); Y = X**2
print("Cov(X, Y)         =", np.cov(X, Y, ddof=0)[0,1])    # ~ 0
print("corr(X, Y)        =", np.corrcoef(X, Y)[0,1])       # ~ 0
print("P(|Y - X^2|<1e-9) =", np.mean(np.abs(Y - X**2) < 1e-9))  # 1.0
```

**5.** A reusable routine:

```python
def joint_moments(P, x_vals, y_vals):
    P = np.asarray(P, float)
    assert np.isclose(P.sum(), 1.0), "P must sum to 1"
    pX = P.sum(0); pY = P.sum(1)
    EX = (x_vals * pX).sum(); EY = (y_vals * pY).sum()
    VarX = ((x_vals - EX)**2 * pX).sum()
    VarY = ((y_vals - EY)**2 * pY).sum()
    EXY = (np.outer(y_vals, x_vals) * P).sum()
    Cov = EXY - EX * EY
    rho = Cov / np.sqrt(VarX * VarY)
    return dict(EX=EX, EY=EY, VarX=VarX, VarY=VarY, Cov=Cov, rho=rho)

# PDF table
print(joint_moments(np.array([[0.10,0.15,0.10],[0.20,0.25,0.20]]),
                    np.array([0,1,2]), np.array([0,1])))
# Independent example: outer(pX, pY)
pX = np.array([0.5, 0.5]); pY = np.array([0.4, 0.6])
P_ind = np.outer(pY, pX)
m = joint_moments(P_ind, pX, pY)
print(m)  # Cov == 0, rho == 0
assert np.isclose(m["Cov"], 0.0)
```


</details>
